# Xarray-Spatial Zonal: Polygon Clipping

Extracting raster data within an irregular boundary is one of the most common GIS operations.
`clip_polygon()` masks pixels outside an arbitrary polygon to a nodata value (NaN by default)
and optionally trims the result to the polygon's bounding box. Unlike `crop()`, which only
extracts a rectangular region, `clip_polygon()` handles watersheds, administrative boundaries,
and any non-rectangular shape.

### What you'll build

1. [Generate a synthetic terrain raster](#Generate-terrain)
2. [Clip to a single polygon](#Basic-clip)
3. [Compare crop=True vs crop=False](#crop=False:-keep-the-original-extent)
4. [Clip with a custom nodata value](#Custom-nodata-value)
5. [Use all_touched to include boundary pixels](#all_touched:-boundary-pixel-inclusion)
6. [Clip from a list of coordinate pairs](#Coordinate-array-input)
7. [Use the .xrs accessor syntax](#Accessor-syntax)

First, import the libraries used throughout this notebook.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import Polygon

from xrspatial import generate_terrain
from xrspatial.polygon_clip import clip_polygon

## Generate terrain

Create a 200x300 synthetic elevation raster to use as the clipping target.

In [ ]:
template = xr.DataArray(np.zeros((200, 300)))
terrain = generate_terrain(template, x_range=(0, 30), y_range=(0, 20))

The terrain raster covers a 30x20 unit area with elevation values generated from
fractal noise. This gives a realistic surface for demonstrating the clip operation.

In [ ]:
terrain.plot.imshow(cmap='terrain', figsize=(10, 6))
plt.title('Synthetic terrain')
plt.gca().set_aspect('equal')
plt.tight_layout()
plt.show()

print(f'Shape: {terrain.shape}')

## Define a clipping polygon

An irregular polygon representing a study area boundary. We draw it on top
of the terrain to see what will be extracted.

In [ ]:
study_area = Polygon([
    (5, 4), (8, 2), (14, 3), (20, 5),
    (22, 10), (18, 16), (12, 17),
    (6, 14), (3, 9),
])

fig, ax = plt.subplots(figsize=(10, 6))
terrain.plot.imshow(ax=ax, cmap='terrain', add_colorbar=True)
patch = MplPolygon(
    list(study_area.exterior.coords),
    closed=True, fill=False, edgecolor='red', linewidth=2,
)
ax.add_patch(patch)
ax.set_title('Terrain with study area boundary')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Basic clip

With `crop=True` (the default), the result is trimmed to the polygon's bounding box.
Pixels outside the polygon are set to NaN.

In [ ]:
clipped = clip_polygon(terrain, study_area)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

terrain.plot.imshow(ax=axes[0], cmap='terrain', add_colorbar=False)
axes[0].set_title('Original')
axes[0].set_aspect('equal')

clipped.plot.imshow(ax=axes[1], cmap='terrain', add_colorbar=False)
axes[1].set_title('Clipped (crop=True)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

print(f'Original shape: {terrain.shape}')
print(f'Clipped shape:  {clipped.shape}')

## crop=False: keep the original extent

When `crop=False`, the output keeps the same shape as the input. Pixels outside the polygon
are still masked to NaN, but the spatial extent stays unchanged. This is useful when you need
the result to align pixel-for-pixel with other rasters.

In [ ]:
clipped_full = clip_polygon(terrain, study_area, crop=False)

clipped_full.plot.imshow(cmap='terrain', figsize=(10, 6))
plt.title('Clipped (crop=False) -- same extent, masked outside polygon')
plt.gca().set_aspect('equal')
plt.tight_layout()
plt.show()

print(f'Shape matches original: {clipped_full.shape == terrain.shape}')

## Custom nodata value

By default, masked pixels become NaN. You can set a different value with the `nodata` parameter.
This is helpful for integer rasters or when downstream tools expect a specific sentinel value.

In [ ]:
clipped_nd = clip_polygon(terrain, study_area, nodata=-9999, crop=False)

# Count cells with each status
vals = clipped_nd.values
n_nodata = np.sum(vals == -9999)
n_valid = np.sum(vals != -9999)
print(f'Valid pixels:  {n_valid}')
print(f'Nodata pixels: {n_nodata}')

## all_touched: boundary pixel inclusion

By default, only pixels whose centre falls inside the polygon are kept. With
`all_touched=True`, any pixel that the polygon boundary touches is also included.
This gives a slightly larger footprint.

In [ ]:
clipped_default = clip_polygon(terrain, study_area, crop=False)
clipped_touched = clip_polygon(terrain, study_area, crop=False, all_touched=True)

n_default = np.count_nonzero(np.isfinite(clipped_default.values))
n_touched = np.count_nonzero(np.isfinite(clipped_touched.values))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

clipped_default.plot.imshow(ax=axes[0], cmap='terrain', add_colorbar=False)
axes[0].set_title(f'all_touched=False ({n_default} pixels)')
axes[0].set_aspect('equal')

clipped_touched.plot.imshow(ax=axes[1], cmap='terrain', add_colorbar=False)
axes[1].set_title(f'all_touched=True ({n_touched} pixels)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

print(f'all_touched adds {n_touched - n_default} boundary pixels')

## Coordinate array input

If you don't have shapely installed in your workflow, you can pass the polygon as a list
of (x, y) coordinate pairs. `clip_polygon()` builds the polygon internally.

In [ ]:
coords = [(5, 4), (8, 2), (14, 3), (20, 5),
          (22, 10), (18, 16), (12, 17), (6, 14), (3, 9)]

clipped_coords = clip_polygon(terrain, coords)

clipped_coords.plot.imshow(cmap='terrain', figsize=(10, 6))
plt.title('Clipped from coordinate list')
plt.gca().set_aspect('equal')
plt.tight_layout()
plt.show()

## Accessor syntax

All xrspatial functions are also available through the `.xrs` accessor on DataArrays.

In [ ]:
import xrspatial  # registers the .xrs accessor

clipped_acc = terrain.xrs.clip_polygon(study_area)
print(f'Accessor result shape: {clipped_acc.shape}')

### References

- [xrspatial.polygon_clip API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.polygon_clip.clip_polygon.html)
- [Shapely Polygon documentation](https://shapely.readthedocs.io/en/stable/reference/shapely.Polygon.html)
- [Rasterio rasterize (similar concept)](https://rasterio.readthedocs.io/en/stable/api/rasterio.features.html#rasterio.features.rasterize)